In [1]:
from ultralytics import YOLO
import cv2
from tqdm import tqdm

model = YOLO("yolo11n-pose.pt")  # load a pretrained model (recommended for training)

video_file = "M:/TDI/GarmentsVision/Camera Visual 2.mp4"


In [2]:
model.predict(source=video_file, show=False, save=True, conf=0.25, device='cuda:0', stream=False )


WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/3683) M:\TDI\GarmentsVision\Camera Visual 2.mp4: 384x640 5 persons, 37.1ms
video 1/1 (frame 2/3683) M:\TDI\GarmentsVision\Camera Visual 2.mp4: 384x640 6 persons, 9.9ms
video 1/1 (frame 3/3683) M:\TDI\GarmentsVision\Camera Visual 2.mp4: 384x640 6 persons, 9.5ms
video 1/1 (frame 4/3683) M:\TDI\GarmentsVision\Camera Visual 2.mp4: 384x640 6 persons, 10.9ms
video 1/1 (frame 5/3683) M:\TDI\GarmentsVision\Camera Visual 2.mp4: 384x640 6 persons, 

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: ultralytics.engine.results.Keypoints object
 masks: None
 names: {0: 'person'}
 obb: None
 orig_img: array([[[124, 134, 138],
         [124, 134, 138],
         [121, 131, 135],
         ...,
         [133, 135, 137],
         [132, 134, 136],
         [132, 134, 136]],
 
        [[124, 134, 138],
         [122, 132, 136],
         [122, 132, 136],
         ...,
         [133, 135, 137],
         [132, 134, 136],
         [132, 134, 136]],
 
        [[122, 132, 136],
         [122, 132, 136],
         [121, 131, 135],
         ...,
         [134, 136, 138],
         [134, 136, 138],
         [133, 135, 137]],
 
        ...,
 
        [[161, 170, 176],
         [160, 169, 175],
         [160, 169, 175],
         ...,
         [ 28,  31,  56],
         [ 28,  34,  58],
         [ 30,  36,  60]],
 
        [[161, 170, 176],
         [160, 169, 175],
         [160, 169,

In [ ]:
import cv2
import torch
from tqdm import tqdm

# --- fast preview knobs (this is a sanity preview, not the main extraction) ---
FRAME_STRIDE = 5      # run detection on every Nth frame
IMGSZ        = 640    # inference resolution (down from native 4K)
DISP_WIDTH   = 1280   # display size; showing full 4K via imshow is a major cost
DEVICE       = 0 if torch.cuda.is_available() else 'cpu'

def show_resized(win, img):
    h = int(img.shape[0] * DISP_WIDTH / img.shape[1])
    cv2.imshow(win, cv2.resize(img, (DISP_WIDTH, h)))

cap = cv2.VideoCapture(video_file)
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
pbar = tqdm(total=total_frames)

idx = 0
try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        idx += 1
        pbar.update(1)

        if idx % FRAME_STRIDE:          # skip frames for a fast preview
            continue

        results = model(frame, conf=0.25, imgsz=IMGSZ, device=DEVICE, verbose=False)[0]
        show_resized("Input Frame", frame)
        show_resized("Annotated Frame", results.plot())

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
finally:
    cap.release()
    cv2.destroyAllWindows()
    pbar.close()

In [ ]:
import cv2
from ultralytics import YOLO
from tqdm import tqdm
import numpy as np

def run_pose_grid_detection(video_path, model_path="yolo11n-pose.pt",
                            rows=3, cols=3, conf=0.25,
                            device='cuda:0', show=True, save=False, output_path="output.mp4"):

    # Load YOLO pose model
    model = YOLO(model_path)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video file: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = cap.get(cv2.CAP_PROP_FPS)
    print(f"Video: {width}x{height}, {fps:.2f} fps, {total_frames} frames")

    
    # Grid dimensions
    cell_w = width // cols
    cell_h = height // rows

    # Optional: setup video writer
    if save:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    else:
        out = None

    pbar = tqdm(total=total_frames, desc="Processing Frames")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Split frame into grid cells
        crops = []
        offsets = []
        i = 0
        for r in range(rows):
            for c in range(cols):
                if not (i==3 or i==4):
                    i+=1
                    continue
                x1 = c * cell_w
                y1 = r * cell_h
                x2 = (c + 1) * cell_w if c < cols - 1 else width
                y2 = (r + 1) * cell_h if r < rows - 1 else height
                crop = frame[y1:y2, x1:x2]
                crops.append(crop)
                offsets.append((x1, y1))
                i+=1
        # Run YOLO pose detection on all crops
        results = model.predict(crops, conf=conf, device=device, verbose=True)
        


        if show:
            for i in range(len(results)):
                cv2.imshow(f"Pose Grid Detection{i}", results[i].plot())
                if cv2.waitKey(1) & 0xFF == ord('q'):  # q to quit
                    break

        if save and out is not None:
            out.write(frame)

        pbar.update(1)

    cap.release()
    if out:
        out.release()
    cv2.destroyAllWindows()
    pbar.close()
    print("✅ Processing complete.")


video_file = "M:/TDI/GarmentsVision/Camera Visual 2.mp4"

# Example usage:
if __name__ == "__main__":
    result = run_pose_grid_detection(
        video_path=video_file,
        model_path="yolo11n-pose.pt",
        rows=3,
        cols=3,
        conf=0.25,
        device='cuda:0',
        show=True,
        save=False,
        output_path="output.mp4"
    )


In [ ]:
import cv2
import time
from ultralytics import YOLO
from tqdm import tqdm
from IPython.display import display, clear_output
from PIL import Image

def run_pose_grid_detection_notebook(video_path,
                                     model_path="yolo11n-pose.pt",
                                     rows=3, cols=3,
                                     target_indices=(3, 4),
                                     conf=0.25, device='cuda:0',
                                     show=True, frame_interval=10):

    model = YOLO(model_path)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video file: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = cap.get(cv2.CAP_PROP_FPS)
    print(f"Video: {width}x{height}, {fps:.2f} fps, {total_frames} frames")

    cell_w = width // cols
    cell_h = height // rows

    pbar = tqdm(total=total_frames, desc="Processing Frames")
    frame_idx = 0

    warmup_done = False

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx += 1

        if frame_idx % frame_interval != 0:
            pbar.update(1)
            continue

        crops = []
        for idx in target_indices:
            r = idx // cols
            c = idx % cols
            x1, y1 = c * cell_w, r * cell_h
            x2 = (c + 1) * cell_w if c < cols - 1 else width
            y2 = (r + 1) * cell_h if r < rows - 1 else height
            crop = frame[y1:y2, x1:x2]
            crops.append((idx, crop))

        annotated_crops = []

        for idx, crop in crops:
            start_time = time.time()
            results = model.predict(crop, conf=conf, device=device, verbose=False)
            elapsed = time.time() - start_time

            annotated = results[0].plot()
            annotated_crops.append(annotated)

            if not warmup_done:
                print(f"✅ Model warmup complete after {elapsed:.2f}s (first crop).")
                warmup_done = True

        if show:
            combined = cv2.hconcat(annotated_crops)
            combined_rgb = cv2.cvtColor(combined, cv2.COLOR_BGR2RGB)
            clear_output(wait=True)
            display(Image.fromarray(combined_rgb))

        pbar.update(1)

    cap.release()
    pbar.close()
    print("✅ Processing complete.")


video_file = "M:/TDI/GarmentsVision/Camera Visual 2.mp4"

run_pose_grid_detection_notebook(
    video_path=video_file,
    model_path="yolo11n-pose.pt",
    rows=3,
    cols=3,
    target_indices=(4,),
    conf=0.25,
    device='cuda:0',
    show=True,
    frame_interval=30  # show every 30th frame for speed
)



In [ ]:
import cv2
from ultralytics import YOLO
from tqdm import tqdm

def init_video_capture(video_path, model_path="yolo11n-pose.pt"):
    """
    Initialize video capture and load YOLO pose model.
    
    Returns:
        cap: cv2.VideoCapture object
        model: YOLO pose model
        total_frames: int
        width: int
        height: int
        fps: float
    """
    # Load YOLO pose model
    model = YOLO(model_path)
    
    # Open video file
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video file: {video_path}")
    
    # Get video properties
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    print(f"Video loaded: {width}x{height}, {fps:.2f} fps, {total_frames} frames")
    
    return cap, model, total_frames, width, height, fps

import matplotlib.pyplot as plt
def crop_frame_custom(frame, x_lines, y_lines):
    """
    Crop a single frame into multiple custom regions based on user-defined lines.
    
    Args:
        frame: np.ndarray, the input frame
        x_lines: list of x coordinates for vertical cuts
        y_lines: list of y coordinates for horizontal cuts
    
    Returns:
        crops: list of cropped frames (np.ndarray)
        crop_boxes: list of dicts with crop_idx and coordinates
    """
    height, width = frame.shape[:2]
    
    # Sort lines and add frame boundaries
    x_sorted = sorted([0] + x_lines + [width])
    y_sorted = sorted([0] + y_lines + [height])
    
    crops = []
    crop_boxes = []
    crop_idx = 0
    
    for i in range(len(y_sorted)-1):
        for j in range(len(x_sorted)-1):
            x1, x2 = x_sorted[j], x_sorted[j+1]
            y1, y2 = y_sorted[i], y_sorted[i+1]
            crop = frame[y1:y2, x1:x2]
            crops.append(crop)
            crop_boxes.append({"crop_idx": crop_idx, "x1": x1, "y1": y1, "x2": x2, "y2": y2})
            crop_idx += 1
    
    return crops, crop_boxes





def plot_crops_grid(crops, cols=3, figsize=(15, 10), titles=None):
    """
    Plot a list of crops in a grid using matplotlib.
    
    Args:
        crops: list of np.ndarray images (BGR)
        cols: number of columns in the grid
        figsize: tuple for figure size
        titles: optional list of titles per crop
    """
    if not crops:
        return
    
    rows = (len(crops) + cols - 1) // cols
    plt.figure(figsize=figsize)
    
    for idx, crop in enumerate(crops):
        plt.subplot(rows, cols, idx + 1)
        # Convert BGR -> RGB
        crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        plt.imshow(crop_rgb)
        plt.axis('off')
        if titles and idx < len(titles):
            plt.title(titles[idx])
    
    plt.tight_layout()
    plt.show()





In [ ]:
import cv2
from ultralytics import YOLO
from tqdm import tqdm
from IPython.display import display, clear_output
from PIL import Image
import numpy as np
import torch
from torchvision.ops import nms




# ----------------------------
# 1️⃣ Video and model setup
# ----------------------------
video_file = "M:/TDI/GarmentsVision/Camera Visual 2.mp4"
cap, model, total_frames, width, height, fps = init_video_capture(video_file, model_path="yolo11n-pose.pt")
pbar = tqdm(total=total_frames)
show = True



while True:
    ret, frame = cap.read()
    if not ret:
        break  # End of video

    # Custom crop lines
    x_lines = [1000, 2500]
    y_lines = [1400, 2100]

    # Get crops
    crops, crop_boxes = crop_frame_custom(frame, x_lines, y_lines)
    crop = crops[4]       # Select a specific crop
    crop_box = crop_boxes[4]

    # Run YOLO pose detection on the crop
    results = model.track(crop, tracker="bytetrack.yaml", conf=0.25, device="cuda:0", verbose=False)

    # Iterate over tracked objects in crop
    for r in results:
        # r.boxes.data: [x1, y1, x2, y2, score, class_id, track_id]
        boxes = r.boxes.data.cpu().numpy()
        keypoints = r.keypoints

        for i, box in enumerate(boxes):
            x1, y1, x2, y2, score, class_id, track_id = box

            # Map to full-frame coordinates
            x1 += crop_box["x1"]
            x2 += crop_box["x1"]
            y1 += crop_box["y1"]
            y2 += crop_box["y1"]

            # Draw bounding box and ID
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
            cv2.putText(frame, f"ID:{int(track_id)}", (int(x1), int(y1)-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

            # Draw pose keypoints
            if keypoints is not None and len(keypoints) > i:
                kpts = keypoints[i].xy.cpu().numpy()  # shape: [num_keypoints, 2]
                # Adjust keypoints to full-frame coordinates
                kpts[:, 0] += crop_box["x1"]
                kpts[:, 1] += crop_box["y1"]
                print(kpts)
                for kx, ky in kpts[0]:
                    cv2.circle(frame, (int(kx), int(ky)), 4, (0, 0, 255), -1)

    # Display in notebook
    if show:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        clear_output(wait=True)
        display(Image.fromarray(frame_rgb))

    pbar.update(1)

cap.release()
pbar.close()

In [ ]:
import cv2
import os
from ultralytics import YOLO
from tqdm import tqdm
import numpy as np
import torch

# ----------------------------
# Config (single source of truth)
# ----------------------------
VIDEO_FILE = "M:/TDI/GarmentsVision/Camera Visual 2.mp4"
MODEL_PATH = "yolo11n-pose.pt"
OUTPUT_DIR = "output"
X_LINES = [1000, 2500]
Y_LINES = [1300, 2100]
CROP_IDX = 4
CONF = 0.5
IOU = 0.7
CONF_THRESHOLD = 0.5
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

os.makedirs(OUTPUT_DIR, exist_ok=True)
POINTS_PATH = os.path.join(OUTPUT_DIR, "hand_points.txt")
VIDEO_OUT_PATH = os.path.join(OUTPUT_DIR, "output_pose_tracking.mp4")

def init_video_capture(video_path, model_path=MODEL_PATH):
    """Initialize video capture and load YOLO pose model."""
    model = YOLO(model_path)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video file: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    print(f"Video loaded: {width}x{height}, {fps:.2f} fps, {total_frames} frames")
    return cap, model, total_frames, width, height, fps

def crop_frame_custom(frame, x_lines, y_lines):
    """Crop a frame into a grid of regions defined by vertical/horizontal cut lines."""
    height, width = frame.shape[:2]
    x_sorted = sorted([0] + x_lines + [width])
    y_sorted = sorted([0] + y_lines + [height])

    crops = []
    crop_boxes = []
    crop_idx = 0
    for i in range(len(y_sorted) - 1):
        for j in range(len(x_sorted) - 1):
            x1, x2 = x_sorted[j], x_sorted[j + 1]
            y1, y2 = y_sorted[i], y_sorted[i + 1]
            crops.append(frame[y1:y2, x1:x2])
            crop_boxes.append({"crop_idx": crop_idx, "x1": x1, "y1": y1, "x2": x2, "y2": y2})
            crop_idx += 1
    return crops, crop_boxes

def draw_pose_keypoints(frame, results, track_ids, conf_threshold=0.5):
    """
    Draw pose keypoints, skeletons, and tracker IDs; collect wrist points.

    COCO-17 pose indices: 9 = LEFT wrist, 10 = RIGHT wrist.
    """
    annotated_frame = frame.copy()
    wrist_points = []

    if results[0].keypoints is not None and results[0].keypoints.conf is not None:
        keypoints = results[0].keypoints.xy.cpu().numpy()
        confidences = results[0].keypoints.conf.cpu().numpy()
        boxes = results[0].boxes.xyxy.cpu().numpy() if results[0].boxes is not None else []

        skeleton = [
            (0, 1), (0, 2), (1, 3), (2, 4),            # Head
            (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),   # Arms
            (5, 11), (6, 12), (11, 12),                # Torso
            (11, 13), (12, 14), (13, 15), (14, 16)     # Legs
        ]

        for person_idx, (kpts, track_id) in enumerate(zip(keypoints, track_ids)):
            person_conf = confidences[person_idx]

            # Keypoints
            for i, (x, y) in enumerate(kpts):
                if person_conf[i] > conf_threshold:
                    cv2.circle(annotated_frame, (int(x), int(y)), 5, (0, 255, 0), -1)

            # Skeleton
            for start_idx, end_idx in skeleton:
                if person_conf[start_idx] > conf_threshold and person_conf[end_idx] > conf_threshold:
                    start_point = (int(kpts[start_idx][0]), int(kpts[start_idx][1]))
                    end_point = (int(kpts[end_idx][0]), int(kpts[end_idx][1]))
                    cv2.line(annotated_frame, start_point, end_point, (255, 0, 0), 2)

            # Tracker ID
            if len(boxes) > person_idx:
                x1, y1, _, _ = boxes[person_idx]
                cv2.putText(annotated_frame, f"ID: {track_id}", (int(x1), int(y1) - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)

            # Wrist points (COCO 9 = left wrist, 10 = right wrist)
            left_wrist = (float(kpts[9][0]), float(kpts[9][1])) if person_conf[9] > conf_threshold else (None, None)
            right_wrist = (float(kpts[10][0]), float(kpts[10][1])) if person_conf[10] > conf_threshold else (None, None)
            wrist_points.append({
                "track_id": track_id,
                "right_wrist": right_wrist,
                "left_wrist": left_wrist
            })

    return annotated_frame, wrist_points

# ----------------------------
# Video and model setup
# ----------------------------
cap, model, total_frames, width, height, fps = init_video_capture(VIDEO_FILE, MODEL_PATH)
print(f"Using device: {DEVICE}, fps: {fps}")
pbar = tqdm(total=total_frames)

# Video writer is opened lazily once we know the true crop size (fixes size mismatch)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = None

# Text file for wrist points
points_file = open(POINTS_PATH, 'w')
points_file.write("Frame,Track_ID,Right_Wrist_X,Right_Wrist_Y,Left_Wrist_X,Left_Wrist_Y\n")

frame_num = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break  # End of video

    # Get crops from the configured ROI grid
    crops, crop_boxes = crop_frame_custom(frame, X_LINES, Y_LINES)
    crop = crops[CROP_IDX]

    # Open writer lazily with the ACTUAL crop dimensions
    if out is None:
        ch, cw = crop.shape[:2]
        out = cv2.VideoWriter(VIDEO_OUT_PATH, fourcc, fps, (cw, ch))

    # ByteTrack-based pose tracking (IDs persist across frames)
    results = model.track(crop, persist=True, tracker="bytetrack.yaml",
                          conf=CONF, iou=IOU, device=DEVICE, verbose=False)

    # Read tracker IDs assigned by ByteTrack
    if results[0].boxes is not None and results[0].boxes.id is not None:
        track_ids = results[0].boxes.id.int().cpu().tolist()
    else:
        track_ids = []

    # Draw pose keypoints, skeletons, IDs, and get wrist points
    annotated_crop, wrist_points = draw_pose_keypoints(crop, results, track_ids, conf_threshold=CONF_THRESHOLD)

    # Write frame to video
    out.write(annotated_crop)

    # Log wrist points to text file
    for person in wrist_points:
        right_x, right_y = person['right_wrist']
        left_x, left_y = person['left_wrist']
        points_file.write(f"{frame_num},{person['track_id']},"
                          f"{right_x if right_x is not None else ''},"
                          f"{right_y if right_y is not None else ''},"
                          f"{left_x if left_x is not None else ''},"
                          f"{left_y if left_y is not None else ''}\n")

    frame_num += 1
    pbar.update(1)

# Release resources
cap.release()
if out is not None:
    out.release()
points_file.close()
pbar.close()
print(f"Output video saved as '{VIDEO_OUT_PATH}'")
print(f"Wrist points saved to '{POINTS_PATH}'")

In [ ]:
import pandas as pd
import numpy as np
import cv2

VIDEO_FILE = "M:/TDI/GarmentsVision/Camera Visual 2.mp4"
DATA_FILE = "output/hand_points.txt"

# Use the TRUE fps / frame count from the source video (no hard-coded 60.0 / 3683)
_cap = cv2.VideoCapture(VIDEO_FILE)
FPS = _cap.get(cv2.CAP_PROP_FPS)
TOTAL_FRAMES = int(_cap.get(cv2.CAP_PROP_FRAME_COUNT))
_cap.release()
print(f"Video: {FPS:.4f} fps, {TOTAL_FRAMES} frames, duration {TOTAL_FRAMES/FPS:.1f}s")

df = pd.read_csv(DATA_FILE)
print("Loaded", df.shape)
print(df['Track_ID'].value_counts().head())
df.head()

In [ ]:
6/61

In [ ]:
# Auto-select the worker tracked for the most frames (robust to ByteTrack IDs)
track_id = df['Track_ID'].value_counts().idxmax()
df_track = df[df['Track_ID'] == track_id].copy().sort_values('Frame')
print(f"Dominant worker: Track_ID {track_id} "
      f"({len(df_track)} / {TOTAL_FRAMES} frames, "
      f"{100*len(df_track)/TOTAL_FRAMES:.1f}% coverage)")
df_track.head()

In [ ]:
def build_series(df_track, col, total_frames, fps):
    """Reindex a per-frame column onto the full 0..total_frames-1 grid and linearly
    interpolate gaps. Avoids the edge-pad step artifact that constant padding injects
    into the FFT."""
    s = (df_track.drop_duplicates('Frame')
                 .set_index('Frame')[col]
                 .reindex(range(total_frames))
                 .interpolate(method='linear', limit_direction='both'))
    t = np.arange(total_frames) / fps
    return t, s.to_numpy()

t, rwx = build_series(df_track, 'Right_Wrist_X', TOTAL_FRAMES, FPS)
_, rwy = build_series(df_track, 'Right_Wrist_Y', TOTAL_FRAMES, FPS)
_, lwx = build_series(df_track, 'Left_Wrist_X',  TOTAL_FRAMES, FPS)
_, lwy = build_series(df_track, 'Left_Wrist_Y',  TOTAL_FRAMES, FPS)
print(f"Built {len(t)} samples spanning {t[-1]:.1f}s "
      f"(NaNs: {int(np.isnan(rwx).sum())})")

In [ ]:
import matplotlib.pyplot as plt
from scipy.signal import detrend

# Drift-free oscillation signal: remove slow positional drift (worker leaning, tracker
# jitter) so the FFT sees the repetitive hand motion rather than the drift.
rwx_dt, lwx_dt = detrend(rwx), detrend(lwx)

fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax[0].plot(t, rwx, label='Right wrist X'); ax[0].plot(t, lwx, label='Left wrist X')
ax[0].set_title('Raw wrist X (oscillation + positional drift)')
ax[0].set_ylabel('pixels'); ax[0].legend(loc='upper right')
ax[1].plot(t, rwx_dt, label='Right wrist X'); ax[1].plot(t, lwx_dt, label='Left wrist X')
ax[1].set_title('Detrended wrist X (oscillation only)')
ax[1].set_xlabel('Time (s)'); ax[1].set_ylabel('pixels'); ax[1].legend(loc='upper right')
plt.tight_layout(); plt.show()

In [ ]:
# 2D wrist trajectories within the crop (sanity view of the motion pattern)
plt.figure(figsize=(7, 5))
plt.plot(rwx, rwy, lw=0.4, alpha=0.7, label='Right wrist')
plt.plot(lwx, lwy, lw=0.4, alpha=0.7, label='Left wrist')
plt.gca().invert_yaxis()  # image coordinates: y grows downward
plt.xlabel('X (px)'); plt.ylabel('Y (px)'); plt.legend()
plt.title('Wrist trajectories (crop image coords)')
plt.tight_layout(); plt.show()

In [ ]:
from scipy.signal import detrend, get_window

def dominant_frequency(signal, fps, fmin=0.05, fmax=1.0, n_peaks=5):
    """Estimate the dominant cyclic frequency of a 1-D signal by FFT peak-picking.

    Detrends (removes drift) and applies a Hann window (reduces spectral leakage),
    then returns the largest spectral peak within [fmin, fmax] Hz -- the frequency is
    DISCOVERED from the data, not assumed in advance.

    Returns: dom_freq, dom_period, freqs, amp, top_freqs, top_amps
    """
    sig = detrend(np.asarray(signal, dtype=float))
    sig = sig * get_window('hann', len(sig))
    amp = np.abs(np.fft.rfft(sig)) * 2.0 / len(sig)        # single-sided amplitude (px)
    freqs = np.fft.rfftfreq(len(sig), d=1.0 / fps)
    band = np.where((freqs >= fmin) & (freqs <= fmax))[0]
    order = band[np.argsort(amp[band])[::-1]]              # peaks, descending amplitude
    top = order[:n_peaks]
    dom = top[0]
    return freqs[dom], 1.0 / freqs[dom], freqs, amp, freqs[top], amp[top]

In [ ]:
# Discover the cycle frequency from the data. Right-wrist horizontal motion is the
# dominant repetitive axis for this workstation.
signal = rwx
dom_freq, dom_period, freqs, amp, top_freqs, top_amps = dominant_frequency(signal, FPS)

dominant_freqs = top_freqs
dominant_amplitudes = top_amps

print(f"Dominant frequency : {dom_freq:.4f} Hz")
print(f"Cycle period       : {dom_period:.2f} s")
print(f"Cycles per minute  : {60 * dom_freq:.1f}")
print("Top peaks (Hz)     :", np.round(top_freqs, 4))
print("Peak amplitudes(px):", np.round(top_amps, 2))

plt.figure(figsize=(10, 4))
band = freqs <= 1.5
plt.plot(freqs[band], amp[band])
plt.axvline(dom_freq, color='r', ls='--',
            label=f'dominant {dom_freq:.3f} Hz  ({dom_period:.1f}s / cycle)')
plt.xlabel('Frequency (Hz)'); plt.ylabel('Amplitude (px)')
plt.title('Spectrum of right-wrist X (detrended + Hann) - data-driven peak')
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

In [ ]:
from scipy.signal import butter, filtfilt

def bandpass_around(signal, fps, f0, frac=0.4):
    """Zero-phase band-pass isolating the cyclic component around the *discovered*
    frequency f0 (not a pre-chosen constant)."""
    sig = detrend(np.asarray(signal, dtype=float))
    lo, hi = f0 * (1 - frac), f0 * (1 + frac)
    b, a = butter(2, [lo / (fps / 2), hi / (fps / 2)], btype='band')
    return filtfilt(b, a, sig)

rwx_band = bandpass_around(rwx, FPS, dom_freq)
lwx_band = bandpass_around(lwx, FPS, dom_freq)

fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax[0].plot(t, detrend(rwx), color='0.75', label='right wrist X (detrended)')
ax[0].plot(t, rwx_band, color='C1', label=f'isolated {dom_freq:.3f} Hz')
ax[0].set_title('Right wrist - cyclic component'); ax[0].set_ylabel('px')
ax[0].legend(loc='upper right')
ax[1].plot(t, detrend(lwx), color='0.75', label='left wrist X (detrended)')
ax[1].plot(t, lwx_band, color='C1', label=f'isolated {dom_freq:.3f} Hz')
ax[1].set_title('Left wrist - cyclic component'); ax[1].set_xlabel('Time (s)')
ax[1].set_ylabel('px'); ax[1].legend(loc='upper right')
plt.tight_layout(); plt.show()

In [ ]:
dominant_amplitudes

In [ ]:
# Final summary (replaces the hard-coded 1/0.025338 magic number)
print(f"Worker Track_ID {track_id}")
print(f"  dominant hand-cycle frequency : {dom_freq:.4f} Hz")
print(f"  cycle period                  : {dom_period:.2f} s")
print(f"  cycles per minute             : {60 * dom_freq:.1f}")